# Data Enrichment
--------------
**Dr. Dave Wanik - University of Connecticut**

Now that we have a geocoded dataset with a zipcode, we can treat the data frame like any other .csv file and join based on a primary key. Later on, we can use this for modeling and see if any of our datasets are useful for prediction.

Note that there is nothing spatial (special) about this notebook - just using your analytics chops to do a merge with multiple dataframes.

--------------------------------------------------------

In [1]:
!pip install ipython-autotime
%load_ext autotime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.0 MB/s eta 0:00:00
time: 516 µs (started: 2024-03-06 16:10:09 +00:00)


In [2]:
from datetime import datetime

now = datetime.now()

current_time = now.strftime("%H:%M:%S")
print("Current Time =", current_time)

Current Time = 16:10:09
time: 2.02 ms (started: 2024-03-06 16:10:09 +00:00)


# Import Modules

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


time: 909 ms (started: 2024-03-06 16:10:09 +00:00)


In [4]:
# this is a GIS version of pandas - perfect for spatial data analysis
!pip install geopandas
import geopandas as gpd

time: 11.7 s (started: 2024-03-06 16:10:10 +00:00)


# Read in the data from before

In [8]:
# https://drive.google.com/file/d/1Ce0T9abhlFYIbArMsgeEreXTxESxRVdq/view?usp=sharing
!gdown 1Ce0T9abhlFYIbArMsgeEreXTxESxRVdq

Downloading...
From: https://drive.google.com/uc?id=1Ce0T9abhlFYIbArMsgeEreXTxESxRVdq
To: /content/airports_zip_latlon_Real_Estate_Sales_2014-2016.csv
100% 408k/408k [00:00<00:00, 111MB/s]
time: 1.11 s (started: 2024-03-06 16:13:08 +00:00)


In [9]:
df = pd.read_csv('/content/airports_zip_latlon_Real_Estate_Sales_2014-2016.csv')

time: 68.7 ms (started: 2024-03-06 16:13:18 +00:00)


# Add zipcode-level data to the housing sales
Look at all of this stuff!

* https://github.com/Ro-Data/Ro-Census-Summaries-By-Zipcode

You can go from census to zipcodes, but it is too involved to show how to do this today (spatial resampling from high-res census tracts to coarser/low-res zip codes).

Someone already did the hard work for you...

![resampling](https://greatdata.com/images/prod/zcta.png)

<center>

![zip codes](https://action-lab.org/youth-sports/files/2019/12/2019-12-04-18_58_09-Hartford-Zip-Code-Map-Showing-Neighborhoods-and-Streets.png)

</center>

Some type of 'weighted average' as a function of people or area is needed...

In [10]:
# warning - these are BIG!!! but not as big as census tracts.
econ = pd.read_table('https://raw.githubusercontent.com/Ro-Data/Ro-Census-Summaries-By-Zipcode/master/econ.txt')
housing = pd.read_table('https://raw.githubusercontent.com/Ro-Data/Ro-Census-Summaries-By-Zipcode/master/housing.txt')
social = pd.read_table('https://raw.githubusercontent.com/Ro-Data/Ro-Census-Summaries-By-Zipcode/master/social.txt')
rural_urban = pd.read_table('https://raw.githubusercontent.com/Ro-Data/Ro-Census-Summaries-By-Zipcode/master/rural_urban.txt')

time: 7.08 s (started: 2024-03-06 16:15:00 +00:00)


In [11]:
print(econ.shape)
print(housing.shape)
print(social.shape)
print(rural_urban.shape)

(33120, 225)
(33120, 255)
(33120, 252)
(33120, 6)
time: 3.3 ms (started: 2024-03-06 16:15:45 +00:00)


In [12]:
print(econ.columns)

Index(['ZCTA5', 'employment_status-population-population_16_years_and_over',
       'employment_status-population-in_labor_force_population_16_years_and_over',
       'employment_status-percent-in_labor_force-of-population_16_years_and_over',
       'employment_status-population-in_civilian_labor_force_population_16_years_and_over',
       'employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over',
       'employment_status-population-employed_population_16_years_and_over',
       'employment_status-percent-employed-of-population_16_years_and_over',
       'employment_status-population-unemployed_population_16_years_and_over',
       'employment_status-percent-unemployed-of-population_16_years_and_over',
       ...
       'people_whose_income_in_past_12m_is_below_poverty_level-percent-people-of-all_people',
       'people_whose_income_in_past_12m_is_below_poverty_level-percent-under_18_years-of-all_people',
       'people_whose_income_in_past_12m_is_below_pover

In [13]:
econ.head()

,ZCTA5,employment_status-population-population_16_years_and_over,employment_status-population-in_labor_force_population_16_years_and_over,employment_status-percent-in_labor_force-of-population_16_years_and_over,employment_status-population-in_civilian_labor_force_population_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over,employment_status-population-employed_population_16_years_and_over,employment_status-percent-employed-of-population_16_years_and_over,employment_status-population-unemployed_population_16_years_and_over,employment_status-percent-unemployed-of-population_16_years_and_over,...,people_whose_income_in_past_12m_is_below_poverty_level-percent-people-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-under_18_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_under_18_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_under_5_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_5_to_17_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-18_years_and_over-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-18_to_64_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-65_years_and_over-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-people_in_families-of-all_families,people_whose_income_in_past_12m_is_below_poverty_level-percent-unrelated_individuals_15_years_and_over-of-all_families
0,601,14169,6056,0.427,6056,0.427,3904,0.276,2152,0.152,...,0.624,0.758,0.758,0.876,0.724,0.583,0.587,0.569,0.613,0.702
1,602,32545,14707,0.452,14676,0.451,11560,0.355,3116,0.096,...,0.545,0.651,0.651,0.794,0.608,0.517,0.535,0.442,0.529,0.664
2,603,41976,16565,0.395,16490,0.393,12722,0.303,3768,0.090,...,0.507,0.645,0.640,0.723,0.614,0.468,0.483,0.421,0.482,0.639
3,606,5160,1672,0.324,1672,0.324,1467,0.284,205,0.040,...,0.640,0.789,0.789,0.918,0.744,0.603,0.657,0.402,0.624,0.738
4,610,22916,9914,0.433,9914,0.433,8327,0.363,1587,0.069,...,0.493,0.621,0.621,0.612,0.623,0.458,0.480,0.384,0.477,0.603


time: 42 ms (started: 2024-03-06 16:17:11 +00:00)


In [14]:
# ensuring that each zipcode is unique
econ['ZCTA5'].nunique()

33120

time: 15.6 ms (started: 2024-03-06 16:17:19 +00:00)


In [15]:
econ['ZCTA5'].dtype

dtype('int64')

time: 5.62 ms (started: 2024-03-06 16:17:30 +00:00)


Now we need to format the `ZCTA5` so that it is a five digit number (with leading 0s if needed.)

In [16]:
econ['ZCTA5'] = econ['ZCTA5'].astype(str).str.zfill(5)
econ.head() # it worked!

,ZCTA5,employment_status-population-population_16_years_and_over,employment_status-population-in_labor_force_population_16_years_and_over,employment_status-percent-in_labor_force-of-population_16_years_and_over,employment_status-population-in_civilian_labor_force_population_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over,employment_status-population-employed_population_16_years_and_over,employment_status-percent-employed-of-population_16_years_and_over,employment_status-population-unemployed_population_16_years_and_over,employment_status-percent-unemployed-of-population_16_years_and_over,...,people_whose_income_in_past_12m_is_below_poverty_level-percent-people-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-under_18_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_under_18_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_under_5_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-related_children_5_to_17_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-18_years_and_over-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-18_to_64_years-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-65_years_and_over-of-all_people,people_whose_income_in_past_12m_is_below_poverty_level-percent-people_in_families-of-all_families,people_whose_income_in_past_12m_is_below_poverty_level-percent-unrelated_individuals_15_years_and_over-of-all_families
0,00601,14169,6056,0.427,6056,0.427,3904,0.276,2152,0.152,...,0.624,0.758,0.758,0.876,0.724,0.583,0.587,0.569,0.613,0.702
1,00602,32545,14707,0.452,14676,0.451,11560,0.355,3116,0.096,...,0.545,0.651,0.651,0.794,0.608,0.517,0.535,0.442,0.529,0.664
2,00603,41976,16565,0.395,16490,0.393,12722,0.303,3768,0.090,...,0.507,0.645,0.640,0.723,0.614,0.468,0.483,0.421,0.482,0.639
3,00606,5160,1672,0.324,1672,0.324,1467,0.284,205,0.040,...,0.640,0.789,0.789,0.918,0.744,0.603,0.657,0.402,0.624,0.738
4,00610,22916,9914,0.433,9914,0.433,8327,0.363,1587,0.069,...,0.493,0.621,0.621,0.612,0.623,0.458,0.480,0.384,0.477,0.603


time: 57.2 ms (started: 2024-03-06 16:18:04 +00:00)


Now do it for the other datasets you created.

In [17]:
housing['ZCTA5'] = housing['ZCTA5'].astype(str).str.zfill(5)
social['ZCTA5'] = social['ZCTA5'].astype(str).str.zfill(5)
rural_urban['ZCTA5'] = rural_urban['ZCTA5'].astype(str).str.zfill(5)

time: 98.3 ms (started: 2024-03-06 16:18:16 +00:00)


Join all four of these datasets together. We will call it `census`.

* https://stackoverflow.com/questions/23668427/pandas-three-way-joining-multiple-dataframes-on-columns

In [18]:
from functools import reduce
dfs = [econ, social, housing, rural_urban]
census = reduce(lambda left,right: pd.merge(left,right,on='ZCTA5'), dfs)
print(census.shape)
census.head()

(33120, 735)


,ZCTA5,employment_status-population-population_16_years_and_over,employment_status-population-in_labor_force_population_16_years_and_over,employment_status-percent-in_labor_force-of-population_16_years_and_over,employment_status-population-in_civilian_labor_force_population_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over,employment_status-population-employed_population_16_years_and_over,employment_status-percent-employed-of-population_16_years_and_over,employment_status-population-unemployed_population_16_years_and_over,employment_status-percent-unemployed-of-population_16_years_and_over,...,gross_rent_as_a_percentage_of_household_income_grapi-housing_units-30_to_less_than_35_percent_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed,gross_rent_as_a_percentage_of_household_income_grapi-percent-30_to_less_than_35_percent-of-occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed,gross_rent_as_a_percentage_of_household_income_grapi-housing_units-35_percent_or_more_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed,gross_rent_as_a_percentage_of_household_income_grapi-percent-35_percent_or_more-of-occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed,gross_rent_as_a_percentage_of_household_income_grapi-housing_units-occupied_units_where_grapi_not_computed,urban_rural-population-total_population,urban_rural-population-urban_total_population,urban_rural-population-rural_total_population,urban_rural-percent-urban_population-of-total_population,urban_rural-percent-rural_population-of-total_population
0,00601,14169,6056,0.427,6056,0.427,3904,0.276,2152,0.152,...,1.09,0.104,5.15,0.490,17.50,18570,10679,7891,0.575,0.425
1,00602,32545,14707,0.452,14676,0.451,11560,0.355,3116,0.096,...,1.37,0.099,5.53,0.400,18.26,41520,41520,0,1.000,0.000
2,00603,41976,16565,0.395,16490,0.393,12722,0.303,3768,0.090,...,2.25,0.057,18.76,0.475,35.66,54689,54646,43,0.999,0.001
3,00606,5160,1672,0.324,1672,0.324,1467,0.284,205,0.040,...,0.06,0.044,0.72,0.526,3.64,6615,2697,3918,0.408,0.592
4,00610,22916,9914,0.433,9914,0.433,8327,0.363,1587,0.069,...,1.03,0.100,5.27,0.514,10.32,29016,25640,3376,0.884,0.116


time: 1.06 s (started: 2024-03-06 16:18:55 +00:00)


These zipcode data can be used for ANY ML modeling problem... think about what you can do with this data...

In [20]:
# we can save a copy of these zipcode datasets
census.to_csv('/content/ZCTA5_censusData.csv')

time: 21.5 s (started: 2024-03-06 16:20:00 +00:00)


Now let's join this to `df_with_zips` - which was our original 100 lat/lon houses.

In [23]:
# prompt: display all columns in pandas dataframe print options

pd.set_option('display.max_columns', None)


time: 817 µs (started: 2024-03-06 16:21:15 +00:00)


In [26]:
# add leading zeros to zipcodes
df['GEOID10'] = df['GEOID10'].astype(str).str.zfill(5)

time: 27.9 ms (started: 2024-03-06 16:22:35 +00:00)


In [27]:
df.head() # GEOID10

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,ID,SerialNumber,ListYear,DateRecorded,Town,Address,AssessedValue,SaleAmount,SalesRatio,PropertyType,ResidentialType,oneAddress,location,point,latitude,longitude,altitude,geometry,index_right,ZCTA5CE10,GEOID10,CLASSFP10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,nearest_geom,nearest_station,line
0,0,0,53330,53331,15045,2015,5/26/2016,Lisbon,26 GRAHAM TERRACE,112970,145000.0,0.779103,Residential,Single Family,26 GRAHAM TERRACE Lisbon CT,"26, Graham Terrace, Lisbon, New London County,...","(41.56667285661604, -72.04098572186558, 0.0)",41.566673,-72.040986,0.0,POINT (-72.04098572186558 41.56667285661604),19711,6351,06351,B5,G6350,S,134731561,7122809,41.589788,-71.948385,POINT (-71.42822111 41.72399917),Theodore F Green State,LINESTRING (-72.04098572186558 41.566672856616...
1,1,61,43605,43606,140265,2014,6/22/2015,Griswold,9-11 HIGH STREET,270130,255000.0,1.059333,Residential,Single Family,9-11 HIGH STREET Griswold CT,"9, High Street, Jewett City, Griswold, New Lon...","(41.6074424802476, -71.97870286076137, 0.0)",41.607442,-71.978703,0.0,POINT (-71.97870286076137 41.6074424802476),19711,6351,06351,B5,G6350,S,134731561,7122809,41.589788,-71.948385,POINT (-71.42822111 41.72399917),Theodore F Green State,LINESTRING (-71.97870286076137 41.607442480247...
2,2,276,53379,53380,14026,2014,4/17/2015,Lisbon,8 SYLVANDALE RD,89700,47352.0,1.894323,Residential,Single Family,8 SYLVANDALE RD Lisbon CT,"8, Sylvandale Road, Lisbon, New London County,...","(41.60114824586855, -71.98849666196837, 0.0)",41.601148,-71.988497,0.0,POINT (-71.98849666196837 41.60114824586855),19711,6351,06351,B5,G6350,S,134731561,7122809,41.589788,-71.948385,POINT (-71.42822111 41.72399917),Theodore F Green State,LINESTRING (-71.98849666196837 41.601148245868...
3,3,437,43433,43434,140190,2014,11/25/2014,Griswold,35 RICHARDSON HILL RD,119140,163000.0,0.730920,Residential,Single Family,35 RICHARDSON HILL RD Griswold CT,"35, Richardson Hill Road, Bethel, Griswold, Ne...","(41.53986221782611, -71.90488915164303, 0.0)",41.539862,-71.904889,0.0,POINT (-71.90488915164303 41.53986221782611),19711,6351,06351,B5,G6350,S,134731561,7122809,41.589788,-71.948385,POINT (-71.42822111 41.72399917),Theodore F Green State,LINESTRING (-71.90488915164303 41.539862217826...
4,4,464,43291,43292,150464,2015,6/17/2016,Griswold,133 POPPLE BRIDGE ROAD,106260,73500.0,1.445714,Residential,Single Family,133 POPPLE BRIDGE ROAD Griswold CT,"133, Popple Bridge Road, Griswold, New London ...","(41.57062184095149, -71.90685918910914, 0.0)",41.570622,-71.906859,0.0,POINT (-71.90685918910914 41.57062184095149),19711,6351,06351,B5,G6350,S,134731561,7122809,41.589788,-71.948385,POINT (-71.42822111 41.72399917),Theodore F Green State,LINESTRING (-71.90685918910914 41.570621840951...


time: 34.3 ms (started: 2024-03-06 16:22:40 +00:00)


In [29]:
df.shape

(731, 34)

time: 32.7 ms (started: 2024-03-06 16:22:59 +00:00)


In [22]:
df.columns

Index(['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'ID', 'SerialNumber',
       'ListYear', 'DateRecorded', 'Town', 'Address', 'AssessedValue',
       'SaleAmount', 'SalesRatio', 'PropertyType', 'ResidentialType',
       'oneAddress', 'location', 'point', 'latitude', 'longitude', 'altitude',
       'geometry', 'index_right', 'ZCTA5CE10', 'GEOID10', 'CLASSFP10',
       'MTFCC10', 'FUNCSTAT10', 'ALAND10', 'AWATER10', 'INTPTLAT10',
       'INTPTLON10', 'nearest_geom', 'nearest_station', 'line'],
      dtype='object')

time: 11.8 ms (started: 2024-03-06 16:20:52 +00:00)


In [28]:
census.head() # let's rename that column for easy merging

,ZCTA5,employment_status-population-population_16_years_and_over,employment_status-population-in_labor_force_population_16_years_and_over,employment_status-percent-in_labor_force-of-population_16_years_and_over,employment_status-population-in_civilian_labor_force_population_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over,employment_status-population-employed_population_16_years_and_over,employment_status-percent-employed-of-population_16_years_and_over,employment_status-population-unemployed_population_16_years_and_over,employment_status-percent-unemployed-of-population_16_years_and_over,employment_status-population-armed_forces_population_16_years_and_over,employment_status-percent-armed_forces-of-population_16_years_and_over,employment_status-population-not_in_labor_force_population_16_years_and_over,employment_status-percent-not_in_labor_force-of-population_16_years_and_over,employment_status-percent-unemployed-of-civilian_labor_force_population_16_years_and_over,employment_status-population_females_16_years_and_over,employment_status-population-in_labor_force_females_16_years_and_over,employment_status-percent-in_labor_force-of-females_16_years_and_over,employment_status-population-in_civilian_labor_force_females_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-females_16_years_and_over,employment_status-population-employed_females_16_years_and_over,employment_status-percent-employed-of-females_16_years_and_over,employment_status-population_own_children_of_the_householder_under_6_years,employment_status-population-all_parents_in_family_in_labor_force_own_children_of_the_householder_under_6_years,employment_status-percent-all_parents_in_family_in_labor_force-of-own_children_of_the_householder_under_6_years,employment_status-population_own_children_of_the_householder_6_to_17_years,employment_status-population-all_parents_in_family_in_labor_force_own_children_of_the_householder_6_to_17_years,employment_status-percent-all_parents_in_family_in_labor_force-of-own_children_of_the_householder_6_to_17_years,commuting_to_work-population_workers_16_years_and_over,commuting_to_work-population-car_truck_or_van_alone_workers_16_years_and_over,commuting_to_work-percent-car_truck_or_van_alone-of-workers_16_years_and_over,commuting_to_work-population-car_truck_or_van_carpooled_workers_16_years_and_over,commuting_to_work-percent-car_truck_or_van_carpooled-of-workers_16_years_and_over,commuting_to_work-population-public_transportation_excluding_taxicab_workers_16_years_and_over,commuting_to_work-percent-public_transportation_excluding_taxicab-of-workers_16_years_and_over,commuting_to_work-population-walked_workers_16_years_and_over,commuting_to_work-percent-walked-of-workers_16_years_and_over,commuting_to_work-population-other_means_workers_16_years_and_over,commuting_to_work-percent-other_means-of-workers_16_years_and_over,commuting_to_work-population-worked_at_home_workers_16_years_and_over,commuting_to_work-percent-worked_at_home-of-workers_16_years_and_over,commuting_to_work-minutes-mean_travel_time_to_work_minutes,occupation-population-civilian_employed_population_16_years_and_over,occupation-population-management_business_science_and_arts_occupations_civilian_employed_population_16_years_and_over,occupation-percent-management_business_science_and_arts_occupations-of-civilian_employed_population_16_years_and_over,occupation-population-service_occupations_civilian_employed_population_16_years_and_over,occupation-percent-service_occupations-of-civilian_employed_population_16_years_and_over,occupation-population-sales_and_office_occupations_civilian_employed_population_16_years_and_over,occupation-percent-sales_and_office_occupations-of-civilian_employed_population_16_years_and_over,occupation-population-natural_resources_construction_and_maintenance_occupations_civilian_employed_population_16_years_and_over,occupation-percent-natural_resources_construction_and_maint

time: 493 ms (started: 2024-03-06 16:22:52 +00:00)


In [30]:
census.rename(columns={'ZCTA5':'GEOID10'}, inplace=True)

time: 2.56 ms (started: 2024-03-06 16:23:09 +00:00)


In [31]:
# make the census GEOID10 column numeric
print(census['GEOID10'].dtype)
print(df['GEOID10'].dtype)

object
object
time: 3.39 ms (started: 2024-03-06 16:23:18 +00:00)


In [32]:
print(df.shape) # 34 columns...
df = df.merge(census, on='GEOID10')
print(df.shape) # enriched data! 768 columns!

(731, 34)
(731, 768)
time: 321 ms (started: 2024-03-06 16:23:25 +00:00)


In [33]:
df.head()

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,ID,SerialNumber,ListYear,DateRecorded,Town,Address,AssessedValue,SaleAmount,SalesRatio,PropertyType,ResidentialType,oneAddress,location,point,latitude,longitude,altitude,geometry,index_right,ZCTA5CE10,GEOID10,CLASSFP10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,nearest_geom,nearest_station,line,employment_status-population-population_16_years_and_over,employment_status-population-in_labor_force_population_16_years_and_over,employment_status-percent-in_labor_force-of-population_16_years_and_over,employment_status-population-in_civilian_labor_force_population_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-population_16_years_and_over,employment_status-population-employed_population_16_years_and_over,employment_status-percent-employed-of-population_16_years_and_over,employment_status-population-unemployed_population_16_years_and_over,employment_status-percent-unemployed-of-population_16_years_and_over,employment_status-population-armed_forces_population_16_years_and_over,employment_status-percent-armed_forces-of-population_16_years_and_over,employment_status-population-not_in_labor_force_population_16_years_and_over,employment_status-percent-not_in_labor_force-of-population_16_years_and_over,employment_status-percent-unemployed-of-civilian_labor_force_population_16_years_and_over,employment_status-population_females_16_years_and_over,employment_status-population-in_labor_force_females_16_years_and_over,employment_status-percent-in_labor_force-of-females_16_years_and_over,employment_status-population-in_civilian_labor_force_females_16_years_and_over,employment_status-percent-in_civilian_labor_force-of-females_16_years_and_over,employment_status-population-employed_females_16_years_and_over,employment_status-percent-employed-of-females_16_years_and_over,employment_status-population_own_children_of_the_householder_under_6_years,employment_status-population-all_parents_in_family_in_labor_force_own_children_of_the_householder_under_6_years,employment_status-percent-all_parents_in_family_in_labor_force-of-own_children_of_the_householder_under_6_years,employment_status-population_own_children_of_the_householder_6_to_17_years,employment_status-population-all_parents_in_family_in_labor_force_own_children_of_the_householder_6_to_17_years,employment_status-percent-all_parents_in_family_in_labor_force-of-own_children_of_the_householder_6_to_17_years,commuting_to_work-population_workers_16_years_and_over,commuting_to_work-population-car_truck_or_van_alone_workers_16_years_and_over,commuting_to_work-percent-car_truck_or_van_alone-of-workers_16_years_and_over,commuting_to_work-population-car_truck_or_van_carpooled_workers_16_years_and_over,commuting_to_work-percent-car_truck_or_van_carpooled-of-workers_16_years_and_over,commuting_to_work-population-public_transportation_excluding_taxicab_workers_16_years_and_over,commuting_to_work-percent-public_transportation_excluding_taxicab-of-workers_16_years_and_over,commuting_to_work-population-walked_workers_16_years_and_over,commuting_to_work-percent-walked-of-workers_16_years_and_over,commuting_to_work-population-other_means_workers_16_years_and_over,commuting_to_work-percent-other_means-of-workers_16_years_and_over,commuting_to_work-population-worked_at_home_workers_16_years_and_over,commuting_to_work-percent-worked_at_home-of-workers_16_years_and_over,commuting_to_work-minutes-mean_travel_time_to_work_minutes,occupation-population-civilian_employed_population_16_years_and_over,occupation-population-management_business_science_and_arts_occupations_civilian_employed_population_16_years_and_over,occupation-percent-management_business_science_and_arts_occupations-of-civilian_employed_population_16_years_and_over,occupation-population-service_occupations_civilian_employed_population_16_years_and_over,occupation-percent-service_occupations-of-civilian_employed_population_16_years_and_over,occupation-population-sales_and_office_oc

time: 603 ms (started: 2024-03-06 16:23:40 +00:00)


# Save Data for Later
This dataset is now BRIMMING with all sorts of good Census data attributes. Let's see if any of it might be useful in a machine learning model.

In [34]:
df.to_csv('/content/census_airports_zip_latlon_Real_Estate_Sales_2014-2016.csv')

time: 788 ms (started: 2024-03-06 16:24:03 +00:00)


Now we just need to split our data into X and Y.... then fit a model.

In [ ]:
from datetime import datetime

later = datetime.now()

later_time = later.strftime("%H:%M:%S")
print("Total Time =", later - now)

Total Time = 0:01:15.859134
time: 3.9 ms (started: 2022-02-24 17:44:15 +00:00)


# Bonus! SweetViz: automated EDA!
Know your data FASTER. Make hundreds and plots and tables with ONE LINE OF CODE.

In [35]:
!pip install sweetviz
import sweetviz as sv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 44.9 MB/s eta 0:00:00
time: 9 s (started: 2024-03-06 16:24:07 +00:00)


In [36]:
# Analyzing data
report=sv.analyze(df, pairwise_analysis='off', target_feat='SaleAmount')

                                             |          | [  0%]   00:00 -> (? left)

/usr/local/lib/python3.10/dist-packages/sweetviz/series_analyzer.py:17: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  value_counts_without_nan = pd.Series()
/usr/local/lib/python3.10/dist-packages/sweetviz/series_analyzer.py:17: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  value_counts_without_nan = pd.Series()
/usr/local/lib/python3.10/dist-packages/sweetviz/series_analyzer.py:17: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  value_counts_without_nan = pd.Series()


time: 27min 11s (started: 2024-03-06 16:24:17 +00:00)


In [37]:
report.show_html()

Report SWEETVIZ_REPORT.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
time: 7.1 s (started: 2024-03-06 16:51:29 +00:00)
